# Alaska SCHISM - full domain and a Cook Inlet subset

Prepares the full Alaska SCHISM mesh, cuts a Cook Inlet subdomain out of it,
and runs both against STOFS boundary forcing.

Everything up to section 5 is preparation and takes a few minutes. The two
simulations are last, so you can build both models and stop there.

Subsetting **after** preparing is deliberate: `extract_mesh` re-keys and
carries over `manning.gr3`, `windrot_geo2proj.gr3`, `nwmReaches.csv`,
`ngenReaches.csv` and the NetCDF mesh, so no prepare command runs twice.

## Setup

The mesh, geogrid and hydrofabric are not in the repo. Create the symlinks
once, pointing at wherever yours live:

```bash
cd docs/examples/alaska_schism
ln -s /path/to/schism_models/alaska           model
ln -s /path/to/geo_em_Alaska.nc               geo_em_Alaska.nc
ln -s /path/to/NWM_v3_hydrofabric.gdb         hydrofabric.gdb
ln -s /path/to/ak_nhf_1.2.2.gpkg              hydrofabric.gpkg
```

Note the Alaska NextGen hydrofabric is the `ak_` prefixed GeoPackage. The
links point at the files themselves, so it does not matter how your copies
are organised.

In [1]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

notebook_dir = Path.cwd()  # assumes run from docs/examples/notebooks/
example_dir = (notebook_dir.parent / "alaska_schism").resolve()
os.chdir(example_dir)

required = (
    "model",
    "geo_em_Alaska.nc",
    "hydrofabric.gdb",
    "hydrofabric.gpkg",
    "run_full.yaml",
    "run_subset.yaml",
)
missing = [n for n in required if not Path(n).exists()]
if missing:
    raise FileNotFoundError(f"Missing in {example_dir}: {missing}. See ../README.md.")


def cli(*args: str) -> None:
    """Run a coastal-calibration CLI command and stream its output."""
    print(f"$ coastal-calibration {' '.join(args)}\n")
    subprocess.run([sys.executable, "-m", "coastal_calibration.cli", *args], check=True)


print(f"Working directory: {example_dir}")
print("Mesh contents:", sorted(p.name for p in Path("model").iterdir()))

Working directory: /home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism
Mesh contents: ['bctides.in', 'hgrid.cpp', 'hgrid.gr3', 'hgrid.ll', 'hgrid.nc', 'manning.gr3', 'ngenReaches.csv', 'nwmReaches.csv', 'open_bnds_hgrid.nc', 'param.nml', 'sflux', 'vgrid.in', 'windrot_geo2proj.gr3']


## 1. Looking at the mesh

Load the mesh with the `nwm_coastal` QGIS plugin. Switching the project to an
Alaska projection makes it easier to orient.

Zoomed in, the extent of the modelled nearshore is visible.

<div style="display: flex; gap: 1em; align-items: flex-start; flex-wrap: wrap;">
  <figure style="flex: 1.43; margin: 0;">
    <img src="../images/Alaska_SCHISM.jpg" alt="Full Alaska SCHISM mesh in QGIS" style="width: 100%;">
    <figcaption>Full Alaska SCHISM mesh in QGIS</figcaption>
  </figure>
  <figure style="flex: 2.11; margin: 0;">
    <img src="../images/Alaska_SCHISM_zoom.jpg" alt="Alaska SCHISM mesh, zoomed" style="width: 100%;">
    <figcaption>Alaska SCHISM mesh, zoomed</figcaption>
  </figure>
</div>

Switch the project back to EPSG:4326 before drawing anything, so polygons are
saved in that CRS. Most tools reproject, but it is the expected one.

![Back to EPSG:4326](../images/Alaska_SCHISM_zoom_EPSG4326.jpg)

The plugin can also write the mesh outline to a GeoJSON, which is handy for
drawing against. `schism_alaska_full_boundary.geojson` in this folder came
from these two steps.

<div style="display: flex; gap: 1em; align-items: flex-start; flex-wrap: wrap;">
  <figure style="flex: 2.36; margin: 0;">
    <img src="../images/Alaska_SCHISM_extract_mesh_boundary.jpg" alt="Extract mesh boundary" style="width: 100%;">
    <figcaption>Extract mesh boundary</figcaption>
  </figure>
  <figure style="flex: 0.89; margin: 0;">
    <img src="../images/Alaska_SCHISM_save_mesh_boundary.jpg" alt="Save mesh boundary" style="width: 100%;">
    <figcaption>Save mesh boundary</figcaption>
  </figure>
</div>

## 2. Prepare the full domain

ESMF mesh files first.

In [2]:
cli("prepare-schism-mesh", "./model", "--force")

$ coastal-calibration prepare-schism-mesh ./model --force



Writing ESMF NetCDF files from hgrid.gr3                                        



Wrote:   /home/laurscham/ngwpc/run_coastal/schism_models/alaska/hgrid.nc
         /home/laurscham/ngwpc/run_coastal/schism_models/alaska/open_bnds_hgrid.nc

  nodes:                 1,188,980
  elements:              2,330,470
  max nodes per element: 3
  open boundary nodes:   53


Open boundaries: 53 nodes in 1 segments                                         
ESMF mesh written: 1,188,980 nodes, 2,330,470 elements, max 3 nodes/element     


`manning.gr3` ships with this mesh. To rebuild it from ESA WorldCover:

```bash
coastal-calibration prepare-schism-manning ./model --force
```

In [3]:
print("manning.gr3 present:", Path("model/manning.gr3").exists())

manning.gr3 present: True


Then the reach crosswalks. Alaska has its own NWM reach layer and its own
NextGen hydrofabric, both picked automatically from the mesh bounds.

In [4]:
cli(
    "prepare-schism-reaches",
    "./model",
    "--nwm-gdb",
    "./hydrofabric.gdb",
    "--ngen-gpkg",
    "./hydrofabric.gpkg",
    "--force",
)

$ coastal-calibration prepare-schism-reaches ./model --nwm-gdb ./hydrofabric.gdb --ngen-gpkg ./hydrofabric.gpkg --force



Inferred hydrofabric region: alaska


nwm_reaches_alaska: 208671 flowline(s) in the mesh bounding box                 


nwm_reaches_alaska: 2686 flowline(s) cross a land boundary                      


nwm_reaches_alaska: 2654 inbound and 567 outbound crossing(s)                   


...

flowpaths: 402 inbound and 181 outbound crossing(s)                             


testing 196303 candidate element(s) near the boundary                           


resolved 583 point(s) inside an element, 0 snapped to the nearest               



Wrote:   /home/laurscham/ngwpc/run_coastal/schism_models/alaska/nwmReaches.csv
  layer:                 nwm_reaches_alaska
  flowlines in bbox:     208,671
  crossing the boundary: 2,686
  inbound crossings:     2,654
  outbound crossings:    567
  merged into an element:69
  resolved by element:   3,221
  resolved by nearest:   0
  source rows:           2,597 (2,455 distinct reaches)
  sink rows:             555 (476 distinct reaches)

Wrote:   /home/laurscham/ngwpc/run_coastal/schism_models/alaska/ngenReaches.csv
  layer:                 flowpaths
  flowlines in bbox:     1,990
  crossing the boundary: 260
  inbound crossings:     402
  outbound crossings:    181
  merged into an element:3
  resolved by element:   583
  resolved by nearest:   0
  source rows:           401 (257 distinct reaches)
  sink rows:             179 (97 distinct reaches)


flowpaths: merged 3 crossing(s) sharing an element                              
wrote 401 source(s) and 179 sink(s) to                                          
/home/laurscham/ngwpc/run_coastal/schism_models/alaska/ngenReaches.csv          


## 3. Draw the subset polygon

Pick an area that makes sense on its own - Cook Inlet here. Roughly
surrounding the area of interest is fine, but where the polygon crosses the
mesh it should cut cleanly, because that edge becomes the subdomain's open
boundary.

Zoomed in on the cut, so the resulting edge is a sensible line rather than a
ragged one.

<div style="display: flex; gap: 1em; align-items: flex-start; flex-wrap: wrap;">
  <figure style="flex: 1.76; margin: 0;">
    <img src="../images/Alaska_SCHISM_subsetpoly.jpg" alt="Cook Inlet subset polygon" style="width: 100%;">
    <figcaption>Cook Inlet subset polygon</figcaption>
  </figure>
  <figure style="flex: 2.37; margin: 0;">
    <img src="../images/Alaska_SCHISM_subsetpoly_zoomwest.jpg" alt="Zoomed on the western cut" style="width: 100%;">
    <figcaption>Zoomed on the western cut</figcaption>
  </figure>
</div>

Saved here as `schism_alaska_subset_cookinlet.geojson`.

## 4. Extract the subset

Keeps the part of the mesh inside the polygon and rebuilds open boundaries
along the cut.

In [5]:
import shapely

import coastal_calibration.schism.subsetter as ss

poly = shapely.get_geometry(
    shapely.from_geojson(Path("schism_alaska_subset_cookinlet.geojson").read_text()), 0
)
print(f"Polygon: {poly.geom_type}, bounds={[round(b, 3) for b in poly.bounds]}")

res = ss.extract_mesh("model", poly, "extracted", output_name="cookinlet")
print(
    f"Extracted: {res.classification.n_side_a:,} nodes, {res.subset.side_a.n_elements:,} elements"
)

Log file: extracted/cookinlet/schism-subset-20260928-061246.log                 


Validated NWM-SCHISM project: model                                             


Polygon: Polygon, bounds=[-154.367, 58.764, -148.652, 61.871]


Classifying nodes inside polygon...                                             


Classification: 520,853 inside, 668,127 outside                                 


...

Copied sflux directory                                                          


Extraction complete: extracted/cookinlet                                        


  520,885 nodes, 1,018,221 elements                                             


  32 shared boundary nodes                                                      


Extracted: 520,885 nodes, 1,018,221 elements


The subset carries the prepared files across, re-keyed to its own element
numbering - nothing needs regenerating.

In [6]:
from coastal_calibration.schism.ngen_reaches import read_reaches_blocks

subset_dir = Path("extracted/cookinlet")
print("subset contents:", sorted(p.name for p in subset_dir.iterdir()))

for name in ("nwmReaches.csv", "ngenReaches.csv"):
    if (subset_dir / name).exists():
        sources, sinks = read_reaches_blocks(subset_dir / name)
        full_s, full_k = read_reaches_blocks(Path("model") / name)
        print(f"{name}: subset {len(sources)}/{len(sinks)} vs full {len(full_s)}/{len(full_k)}")

subset contents: ['bctides.in', 'element_areas.txt', 'hgrid.cpp', 'hgrid.gr3', 'hgrid.ll', 'hgrid.nc', 'manning.gr3', 'ngenReaches.csv', 'node_mapping.txt', 'nwmReaches.csv', 'open_bnds_hgrid.nc', 'param.nml', 'schism-subset-20260927-052845.log', 'schism-subset-20260927-061245.log', 'schism-subset-20260928-061246.log', 'sflux', 'vgrid.in', 'windrot_geo2proj.gr3']
nwmReaches.csv: subset 1408/365 vs full 2597/555
ngenReaches.csv: subset 213/86 vs full 401/179


### hgrid.cpp

This mesh did not ship an `hgrid.cpp`. SCHISM's own solver never reads one, but the
`combine_sink_source` utility it runs during the discharge stage opens it
unconditionally, and aborts when it is absent.

In the mode that utility is called with it uses the element connectivity and the
boundary block, both of which `hgrid.gr3` already carries, so a link to that file
satisfies it. Note this writes into the linked mesh directory, not into the example.

In [7]:
for d in (Path("model"), Path("extracted/cookinlet")):
    cpp = d / "hgrid.cpp"
    if not cpp.is_symlink() and not cpp.exists():
        cpp.symlink_to("hgrid.gr3")
    print(f"{cpp}: -> {cpp.readlink() if cpp.is_symlink() else 'existing file'}")

model/hgrid.cpp: -> hgrid.gr3
extracted/cookinlet/hgrid.cpp: -> hgrid.gr3


## 5. Run

Both configs are validated first. The subset is much smaller, so run it
first.

In [8]:
cli("run", "run_subset.yaml", "--dry-run")
cli("run", "run_full.yaml", "--dry-run")

$ coastal-calibration run run_subset.yaml --dry-run



Dry run mode - validating configuration...                                      
Dry run mode - validation passed, no execution                                  
Workflow completed successfully.                                                


$ coastal-calibration run run_full.yaml --dry-run



Dry run mode - validating configuration...                                      
Dry run mode - validation passed, no execution                                  
Workflow completed successfully.                                                


`nodes` and `ntasks_per_node` in each yaml size the MPI job, so the same
command works on a workstation and on the cluster.

In [9]:
print(Path("run_subset.yaml").read_text())

# Alaska SCHISM - subset domain run configuration
#
# Mesh reached through the ./model symlink; see ../README.md. The subset domain is
# prepared in ../notebooks/alaska_schism.ipynb before this config is used.
#
# Usage:
#   coastal-calibration run run_subset.yaml --dry-run
#   coastal-calibration run run_subset.yaml

model: schism

simulation:
  start_date: 2026-08-15 00:00:00
  duration_hours: 12
  coastal_domain: alaska
  meteo_source: nwm_ana

boundary:
  source: stofs        # STOFS 2D Global covers Alaska

paths:
  work_dir: ./run_subset
  raw_download_dir: ../downloads

download:
  enabled: true

model_config:
  prebuilt_dir: ./extracted/cookinlet
  geogrid_file: ./geo_em_Alaska.nc
  nodes: 1
  ntasks_per_node: 16
  include_noaa_gages: true
  run_param_overrides:
    ic_elev: 0 # prebuilt schism model did not come with elev.ic file, turning off



### Cook Inlet subset

In [10]:
cli("run", "run_subset.yaml")

$ coastal-calibration run run_subset.yaml



Coastal Calibration Workflow                                                    
Start Time: 2026-09-28 06:13:49                                                 
----------------------------------------                                        
Cleaned generated files from                                                    
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_subset        
Stage: download                                                                 
Start Time: 2026-09-28 06:13:49                                                 
  Download input data (NWM, STOFS)                                              


STOFS: using cycle 2026-08-15 00:00:00 for start 2026-08-15 00:00:00            


  meteo/nwm_ana: 13/13 [OK]                                                     
  hydro/nwm: 13/13 [OK]                                                         
  coastal/stofs: 1/1 [OK]                                                       
  Total: 27/27 (failed: 0)                                                      
  Download complete. Raw files stored in                                        
/home/laurscham/ngwpc/nwm-coastal/docs/examples/downloads                       
  [✓] COMPLETED (4m 43s)                                                        
----------------------------------------                                        
Stage: schism_forcing_prep                                                      
Start Time: 2026-09-28 06:18:33                                                 
  Prepare LDASIN forcing data                                                   
  Creating forcing symlinks and output directories                              
    Staged LDASIN files in  

  NWM forcing generated in                                                      
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_subset/forcing
_input                                                                          
  [✓] COMPLETED (19s)                                                           
----------------------------------------                                        
Stage: schism_sflux                                                             
Start Time: 2026-09-28 06:18:52                                                 
  Generate sflux atmospheric files                                              
  Generating sflux from LDASIN files                                            
    No hgrid.ll or hgrid.gr3 in                                                 
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_subset;       
writing full geogrid (no subsetting).                                           
    Creating sflux from 13 L

...

  SCHISM run completed successfully (output:                                    
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_subset/outputs
/schism_log.txt)                                                                
  [✓] COMPLETED (27m 43s)                                                       
----------------------------------------                                        
Stage: schism_postprocess                                                       
Start Time: 2026-09-28 06:46:57                                                 
  Post-process SCHISM outputs                                                   
    No hotstart_000000_*.nc files found in                                      
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_subset/outputs
; nothing to combine                                                            
  SCHISM post-processing complete                                               
  [✓] COMPLETED (0s)        

  Successfully retrieved data for 3 station(s)                                  


  Loading cached station metadata from cache/coops_stations_metadata.json       
  Fetching datum information for 3 station(s)                                   
  Fetching data from 3 station(s)                                               


  Saved 1 comparison figure(s) to                                               
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_subset/figs   
  Including 3 NOAA gauge(s) from station.in                                     


  Wrote obs_water_level.parquet with 3 station(s) and 12 timestep(s)            
  [✓] COMPLETED (3s)                                                            
----------------------------------------                                        
Workflow COMPLETED | Total Duration: 0:33:12                                    
                                                                                
Timing Summary:                                                                 
----------------------------------------                                        
  [✓] download: 4m 43s                                                          
  [✓] schism_forcing_prep: 0s                                                   
  [✓] schism_forcing: 19s                                                       
  [✓] schism_sflux: 0s                                                          
  [✓] schism_params: 0s                                                         
  [✓] schism_obs: 2s        

### Full domain

The full Alaska mesh is a much bigger job - `run_full.yaml` asks for 4 nodes.

In [11]:
cli("run", "run_full.yaml")

$ coastal-calibration run run_full.yaml



Coastal Calibration Workflow                                                    
Start Time: 2026-09-28 06:47:04                                                 
----------------------------------------                                        
Cleaned generated files from                                                    
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_full          
Stage: download                                                                 
Start Time: 2026-09-28 06:47:04                                                 
  Download input data (NWM, STOFS)                                              
  meteo/nwm_ana: 13/13 [OK]                                                     
  hydro/nwm: 13/13 [OK]                                                         
  coastal/stofs: 1/1 [OK]                                                       
  Total: 27/27 (failed: 0)                                                      
  Download complete. Raw fil

  NWM forcing generated in                                                      
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_full/forcing_i
nput                                                                            
  [✓] COMPLETED (24s)                                                           
----------------------------------------                                        
Stage: schism_sflux                                                             
Start Time: 2026-09-28 06:47:29                                                 
  Generate sflux atmospheric files                                              
  Generating sflux from LDASIN files                                            
    No hgrid.ll or hgrid.gr3 in                                                 
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_full; writing 
full geogrid (no subsetting).                                                   
    Creating sflux from 13 L

    Renamed sflux_air_1.0001.nc → sflux_air_1.1.nc                              
    Generated sflux in                                                          
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_full/sflux    
  Post-processing complete. 1 sflux file(s) produced.                           
  [✓] COMPLETED (0s)                                                            
----------------------------------------                                        
Stage: schism_params                                                            
Start Time: 2026-09-28 06:47:29                                                 
  Create param.nml and symlink mesh files                                       
  Creating param.nml and symlinking mesh files                                  
    Created param.nml and symlinked mesh files in                               
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_full          
  Parameter file created:   

  hgrid.gr3: 1188980 nodes, 2330470 elements                                    


...

  SCHISM run completed successfully (output:                                    
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_full/outputs/s
chism_log.txt)                                                                  
  [✓] COMPLETED (1h 17m 50s)                                                    
----------------------------------------                                        
Stage: schism_postprocess                                                       
Start Time: 2026-09-28 08:06:25                                                 
  Post-process SCHISM outputs                                                   
    No hotstart_000000_*.nc files found in                                      
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_full/outputs; 
nothing to combine                                                              
  SCHISM post-processing complete                                               
  [✓] COMPLETED (0s)        

  Successfully retrieved data for 5 station(s)                                  


  Loading cached station metadata from cache/coops_stations_metadata.json       
  Fetching datum information for 5 station(s)                                   
  Fetching data from 5 station(s)                                               


  Saved 2 comparison figure(s) to                                               
/home/laurscham/ngwpc/nwm-coastal/docs/examples/alaska_schism/run_full/figs     
  Including 5 NOAA gauge(s) from station.in                                     


  Wrote obs_water_level.parquet with 5 station(s) and 12 timestep(s)            
  [✓] COMPLETED (5s)                                                            
----------------------------------------                                        
Workflow COMPLETED | Total Duration: 1:19:26                                    
                                                                                
Timing Summary:                                                                 
----------------------------------------                                        
  [✓] download: 0s                                                              
  [✓] schism_forcing_prep: 0s                                                   
  [✓] schism_forcing: 24s                                                       
  [✓] schism_sflux: 0s                                                          
  [✓] schism_params: 0s                                                         
  [✓] schism_obs: 4s        

## Summary

Prepare once per mesh, subset from the prepared mesh, then run. The same
three `prepare-schism-*` commands apply to any SCHISM domain; only the
hydrofabric paths and `coastal_domain` change.